# Query BV-BRC for The Seed annotation

## Get IDs for strain

First we need to get the internal ID for the strain of interest. Here as an example, search for Staphylococcus areus Newman:

In [4]:
import requests
import pandas as pd

In [5]:
URL = "https://www.bv-brc.org/api/genome/"

terms = ["Staphylococcus", "aureus", "Newman"]
conds = ",".join(f"eq(genome_name,*{t}*)" for t in terms)
query = f"and({conds})&select(genome_id,genome_name,taxon_id)&limit(1000)"

r = requests.get(f"{URL}?{query}", headers={"Accept": "application/json"})
r.raise_for_status()

genomes = pd.DataFrame(r.json())
print(genomes)

     genome_id                                      genome_name  taxon_id
0   1280.53126                   Staphylococcus aureus NewmanDt      1280
1   1280.55094                 Staphylococcus aureus Newman HOM      1280
2   1280.55093             Staphylococcus aureus Newman D2C HOM      1280
3    426430.67  Staphylococcus aureus subsp. aureus str. Newman    426430
4    426430.63  Staphylococcus aureus subsp. aureus str. Newman    426430
5     426430.8  Staphylococcus aureus subsp. aureus str. Newman    426430
6   1280.10152       Staphylococcus aureus strain Sa_Newman_UoM      1280
7   1280.37689        Staphylococcus aureus strain Newman NM-CQ      1280
8   1280.49819                  Staphylococcus aureus Newman_Dt      1280
9   1280.49820                  Staphylococcus aureus Newman_WT      1280
10   426430.24  Staphylococcus aureus subsp. aureus str. Newman    426430


Next determine for which `genome_id`s RefSeq annotations are available:

In [6]:
URL = "https://www.bv-brc.org/api/genome_feature/"

ids = ",".join(genomes["genome_id"])
# keep only RefSeq features, then facet on genome_id to get one count per genome
query = (
    f"and(in(genome_id,({ids})),eq(annotation,RefSeq))"
    "&limit(1)"
    "&facet((field,genome_id),(mincount,1))"
)

r = requests.get(f"{URL}?{query}", headers={"Accept": "application/solr+json"})
r.raise_for_status()

# Solr returns facets as a flat list: [id1, count1, id2, count2, ...]
facet = r.json()["facet_counts"]["facet_fields"]["genome_id"]
counts = dict(zip(facet[::2], facet[1::2]))

refseq_counts = genomes.assign(
    n_refseq_features=genomes["genome_id"].map(counts).fillna(0).astype(int)
)
print(refseq_counts)

     genome_id                                      genome_name  taxon_id  \
0   1280.53126                   Staphylococcus aureus NewmanDt      1280   
1   1280.55094                 Staphylococcus aureus Newman HOM      1280   
2   1280.55093             Staphylococcus aureus Newman D2C HOM      1280   
3    426430.67  Staphylococcus aureus subsp. aureus str. Newman    426430   
4    426430.63  Staphylococcus aureus subsp. aureus str. Newman    426430   
5     426430.8  Staphylococcus aureus subsp. aureus str. Newman    426430   
6   1280.10152       Staphylococcus aureus strain Sa_Newman_UoM      1280   
7   1280.37689        Staphylococcus aureus strain Newman NM-CQ      1280   
8   1280.49819                  Staphylococcus aureus Newman_Dt      1280   
9   1280.49820                  Staphylococcus aureus Newman_WT      1280   
10   426430.24  Staphylococcus aureus subsp. aureus str. Newman    426430   

    n_refseq_features  
0                2939  
1                2955  
2  

In [7]:
URL = "https://www.bv-brc.org/api/genome_feature/"
genome_id = "426430.67"
fields = ["feature_id", "patric_id", "refseq_locus_tag", "protein_id"]

# page through the results; a stable sort keeps the pages consistent
page_size = 10000
rows, offset = [], 0
while True:
    query = (
        f"eq(genome_id,{genome_id})"
        f"&select({','.join(fields)})"
        "&sort(+feature_id)"
        f"&limit({page_size},{offset})"
    )
    r = requests.get(f"{URL}?{query}", headers={"Accept": "application/json"})
    r.raise_for_status()
    batch = r.json()
    rows.extend(batch)
    if len(batch) < page_size:
        break
    offset += page_size

features = pd.DataFrame(rows, columns=fields)
print(features)

                                             feature_id  \
0     PATRIC.426430.67.JBLSAI010000001.CDS.100434.10...   
1     PATRIC.426430.67.JBLSAI010000001.CDS.101744.10...   
2     PATRIC.426430.67.JBLSAI010000001.CDS.102845.10...   
3     PATRIC.426430.67.JBLSAI010000001.CDS.103585.10...   
4     PATRIC.426430.67.JBLSAI010000001.CDS.105807.10...   
...                                                 ...   
6000  RefSeq.426430.67.JBLSAI010000060.source.1.302.fwd   
6001  RefSeq.426430.67.JBLSAI010000061.rRNA.174.288.rev   
6002  RefSeq.426430.67.JBLSAI010000061.source.1.294.fwd   
6003     RefSeq.426430.67.JBLSAI010000062.CDS.1.292.fwd   
6004  RefSeq.426430.67.JBLSAI010000062.source.1.292.fwd   

                  patric_id refseq_locus_tag    protein_id  
0     fig|426430.67.peg.104     ACNFN8_00520           NaN  
1     fig|426430.67.peg.105     ACNFN8_00525           NaN  
2     fig|426430.67.peg.106     ACNFN8_00530           NaN  
3     fig|426430.67.peg.107     ACNFN8_00535   

Select one of the genomes with many RefSeq annotations. Get the information for all annotations with a `protein_id`

## Map locus tags to RefSeq `WP_` protein IDs

BV-BRC only has the GenBank protein IDs (`MEJ…`). The RefSeq `WP_` IDs are in NCBI's RefSeq copy of the contigs (`NZ_…`). Fetch the feature tables of all contigs via E-utilities and link each gene's `old_locus_tag` (= BV-BRC `refseq_locus_tag`) to the `WP_` ID of its CDS.

In [8]:
BVBRC = "https://www.bv-brc.org/api"
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# 1) contig accessions from BV-BRC (GenBank names, e.g. JBBIAF010000001)
r = requests.get(
    f"{BVBRC}/genome_sequence/?eq(genome_id,{genome_id})&select(accession)&limit(10000)",
    headers={"Accept": "application/json"},
)
r.raise_for_status()
refseq_acc = ["NZ_" + d["accession"] for d in r.json()]  # RefSeq copies have an NZ_ prefix

# 2) feature tables of all contigs in one POST request
r = requests.post(f"{EUTILS}/efetch.fcgi", data={
    "db": "nuccore", "id": ",".join(refseq_acc), "rettype": "ft", "retmode": "text",
})
r.raise_for_status()

# 3) parse: ">Feature" = new contig, "start\tend\ttype" = new feature,
#    "\t\t\tkey\tvalue" = qualifier of the current feature
rows, contig, feat = [], None, None
for line in r.text.splitlines():
    if line.startswith(">Feature"):
        contig = line.split("|")[1]
        continue
    cols = line.split("\t")
    if len(cols) >= 3 and cols[2]:
        feat = {"contig": contig, "start": cols[0], "end": cols[1], "type": cols[2]}
        rows.append(feat)
    elif len(cols) >= 5 and feat is not None:
        feat.setdefault(cols[3], cols[4])  # keep first value of repeated qualifiers

ft = pd.DataFrame(rows)

# 4) one row per gene, with the WP_ ID of its CDS (gene and CDS share coordinates)
# TODO: matching is not perfect for peptide chain release factor 2 (prfB, WP_010956549.1)
# Its CDS needs a ribosomal frameshift: the CDS location is made of two pieces, while the
# gene is one continuous range. The parser only reads the first location line, so the CDS's
# end (259023) differs from the gene's, and the gene ends up without a WP_ ID.
# alternatives:
# - Match on contig + start only. Both pieces start at the same position, but that
# relies on start positions being unique, which is less safe.
# - Match on order. Give each CDS the locus tag of the gene directly before it in the file,
# which is how the feature table is laid out.
genes = ft[ft["type"] == "gene"][["contig", "start", "end", "locus_tag", "old_locus_tag"]]
cds = ft[ft["type"] == "CDS"][["contig", "start", "end", "protein_id", "product"]]
cds = cds.assign(wp_id=cds["protein_id"].str.extract(r"(WP_[\d.]+)", expand=False))

locus_map = genes.merge(cds.drop(columns="protein_id"), on=["contig", "start", "end"], how="left")
print(locus_map)

                    contig   start      end       locus_tag old_locus_tag  \
0     NZ_JBLSAI010000035.1      <1     >806  ACNFN8_RS14415  ACNFN8_14405   
1     NZ_JBLSAI010000042.1      <1      155  ACNFN8_RS14475  ACNFN8_14465   
2     NZ_JBLSAI010000042.1     196      495  ACNFN8_RS14480  ACNFN8_14470   
3     NZ_JBLSAI010000053.1    <387       >1  ACNFN8_RS14545  ACNFN8_14535   
4     NZ_JBLSAI010000012.1      <1      121  ACNFN8_RS13520  ACNFN8_13510   
...                    ...     ...      ...             ...           ...   
2913  NZ_JBLSAI010000006.1  105799   107949  ACNFN8_RS11850  ACNFN8_11840   
2914  NZ_JBLSAI010000006.1  107942   108397  ACNFN8_RS11855  ACNFN8_11845   
2915  NZ_JBLSAI010000006.1  109044   109126  ACNFN8_RS11860  ACNFN8_11850   
2916  NZ_JBLSAI010000006.1  109149   109222  ACNFN8_RS11865  ACNFN8_11855   
2917  NZ_JBLSAI010000006.1  109341  >109419  ACNFN8_RS11870  ACNFN8_11860   

                                product           wp_id  
0         phage b

In [9]:
# add the RefSeq locus tag and WP_ ID to the BV-BRC features
# (drop genes without old_locus_tag first: pandas would match NaN to NaN)
# only keep entries with a patric_id and 
features_wp = (
    features.merge(
        locus_map.dropna(subset="old_locus_tag")[["old_locus_tag", "locus_tag", "wp_id", "product"]],
        left_on="refseq_locus_tag",
        right_on="old_locus_tag",
        how="left",
    )
    .drop(columns="old_locus_tag")
    .dropna(subset=["patric_id", "wp_id"])
)
print(features_wp)

                                             feature_id  \
0     PATRIC.426430.67.JBLSAI010000001.CDS.100434.10...   
1     PATRIC.426430.67.JBLSAI010000001.CDS.101744.10...   
2     PATRIC.426430.67.JBLSAI010000001.CDS.102845.10...   
3     PATRIC.426430.67.JBLSAI010000001.CDS.103585.10...   
4     PATRIC.426430.67.JBLSAI010000001.CDS.105807.10...   
...                                                 ...   
2967   PATRIC.426430.67.JBLSAI010000034.CDS.672.851.rev   
2969   PATRIC.426430.67.JBLSAI010000036.CDS.137.742.rev   
2980   PATRIC.426430.67.JBLSAI010000042.CDS.196.495.fwd   
2981     PATRIC.426430.67.JBLSAI010000042.CDS.3.155.fwd   
2987     PATRIC.426430.67.JBLSAI010000047.CDS.1.260.fwd   

                   patric_id refseq_locus_tag protein_id       locus_tag  \
0      fig|426430.67.peg.104     ACNFN8_00520        NaN  ACNFN8_RS00520   
1      fig|426430.67.peg.105     ACNFN8_00525        NaN  ACNFN8_RS00525   
2      fig|426430.67.peg.106     ACNFN8_00530        NaN  ACNFN

In [26]:
# save the mapping information

features_wp.to_csv("../data/example_features_wp.tsv", sep="\t", index=False)

## Get The Seed information

In [10]:
URL = "https://www.bv-brc.org/api/subsystem/"
fields = [
    "patric_id", "feature_id", "refseq_locus_tag", "gene", "product",
    "superclass", "class", "subclass", "subsystem_id", "subsystem_name",
    "role_id", "role_name", "active",
]

# page through the results; a stable sort keeps the pages consistent
page_size = 10000
rows, offset = [], 0
while True:
    query = (
        f"eq(genome_id,{genome_id})"
        f"&select({','.join(fields)})"
        "&sort(+id)"
        f"&limit({page_size},{offset})"
    )
    r = requests.get(f"{URL}?{query}", headers={"Accept": "application/json"})
    r.raise_for_status()
    batch = r.json()
    rows.extend(batch)
    if len(batch) < page_size:
        break
    offset += page_size

subsystems = pd.DataFrame(rows, columns=fields)
print(subsystems)

                   patric_id  \
0      fig|426430.67.peg.353   
1     fig|426430.67.peg.2216   
2      fig|426430.67.peg.506   
3     fig|426430.67.peg.2045   
4      fig|426430.67.peg.579   
...                      ...   
1379   fig|426430.67.peg.977   
1380  fig|426430.67.peg.2477   
1381   fig|426430.67.peg.977   
1382  fig|426430.67.peg.2052   
1383  fig|426430.67.peg.2381   

                                             feature_id refseq_locus_tag  \
0     PATRIC.426430.67.JBLSAI010000001.CDS.389199.38...     ACNFN8_01750   
1     PATRIC.426430.67.JBLSAI010000004.CDS.215321.21...     ACNFN8_09825   
2     PATRIC.426430.67.JBLSAI010000001.CDS.532122.53...     ACNFN8_02515   
3     PATRIC.426430.67.JBLSAI010000004.CDS.62334.625...     ACNFN8_08980   
4     PATRIC.426430.67.JBLSAI010000001.CDS.604672.60...     ACNFN8_02885   
...                                                 ...              ...   
1379  PATRIC.426430.67.JBLSAI010000002.CDS.40236.413...     ACNFN8_03835   
1380  P

In [11]:
column_order = ["wp_id", "refseq_locus_tag", "gene", "product", "superclass", "class", "subclass", "subsystem_id", "subsystem_name", "role_id", "role_name"]
final_data = (
    subsystems.merge(
        features_wp[["patric_id", "wp_id"]], on="patric_id", how="left"
        )
    .dropna(subset="wp_id")
    .reindex(columns=column_order)
    )


In [12]:
print(final_data)

               wp_id refseq_locus_tag  gene  \
0     WP_001108889.1     ACNFN8_01750  recU   
1     WP_000606546.1     ACNFN8_09825  nreB   
2     WP_000579275.1     ACNFN8_02515  mtnN   
3     WP_001118443.1     ACNFN8_08980  infA   
4     WP_001138360.1     ACNFN8_02885  rplT   
...              ...              ...   ...   
1379  WP_000035320.1     ACNFN8_03835  pdhA   
1380  WP_001021214.1     ACNFN8_11080   NaN   
1381  WP_000035320.1     ACNFN8_03835  pdhA   
1382  WP_000091975.1     ACNFN8_09015  rplF   
1383  WP_001045079.1     ACNFN8_10590   NaN   

                                                product  \
0                      RecU Holliday junction resolvase   
1     Oxygen-sensing two-component system sensor his...   
2     5'-methylthioadenosine nucleosidase (EC 3.2.2....   
3                       Translation initiation factor 1   
4                            LSU ribosomal protein L20p   
...                                                 ...   
1379  Pyruvate dehydro

## Example data

Test how well it annotates.

In [13]:
test_data = pd.ExcelFile("../../Example_Staphylococcus_aureus_Newman_TaxId_426430.xlsx").parse("Tabelle1")

/var/folders/z3/xb62blvs5xg7hy35ktdz1mdw0000gn/T/ipykernel_30060/911866719.py:1: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  test_data = pd.ExcelFile("../../Example_Staphylococcus_aureus_Newman_TaxId_426430.xlsx").parse("Tabelle1")


In [15]:
test_data = test_data.assign(wp_id=test_data["Majority protein IDs"].str.extract(r"(WP_[\d.]+)", expand=False))

In [17]:
test_data = test_data.merge(final_data, how="left", on="wp_id")

In [18]:
test_data

,Majority protein IDs,NWMN neu,NWMN alt,gen symbol,Annotation,TIGRFAM Sub Role,The SEED,wp_id,refseq_locus_tag,gene,product,superclass,class,subclass,subsystem_id,subsystem_name,role_id,role_name
0,WP_000942300.1|NWMN_RS07340|NWMN_1303|,NWMN_RS07340,NWMN_1303,NWMN_1303,ATP-binding cassette domain-containing protei...,0,NaN,WP_000942300.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,WP_000602071.1|NWMN_RS11270|NWMN_1954|,NWMN_RS11270,NWMN_1954,NWMN_1954,ABC-F type ribosomal protection protein[Staph...,0,NaN,WP_000602071.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,WP_000524836.1|NWMN_RS12920|NWMN_2242|,NWMN_RS12920,NWMN_2242,NWMN_2242,ABC transporter ATP-binding protein[Staphyloc...,Other,NaN,WP_000524836.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,WP_001229077.1|NWMN_RS13625|NWMN_2364|,NWMN_RS13625,NWMN_2364,opp1A,staphylopine-dependent metal ABC transporter ...,Cations and iron carrying compounds,"""ABC transporter dipeptide (TC 3.A.1.5.2)""",WP_001229077.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,WP_001172340.1|NWMN_RS14380|NWMN_2506|,NWMN_RS14380,NWMN_2506,NWMN_2506,AMP-binding protein[Staphylococcus aureus sub...,0,"AMP-(fatty) acid ligases, YtcI homolog""",WP_001172340.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283,WP_000872653.1|NWMN_RS09020|NWMN_1609|,NWMN_RS09020,NWMN_1609,thiI,tRNA uracil 4-sulfurtransferase ThiI[Staphylo...,Thiamine; tRNA and rRNA base modification,"""Thiamin biosynthesis""",WP_000872653.1,ACNFN8_03075,thiI,tRNA 4-thiouridine synthase (EC 2.8.1.4),RNA PROCESSING,RNA Processing,RNA processing and modification,tRNA_thiolation,tRNA thiolation,tRNA_4-thiouridine_synthase_(EC_2.8.1.4),tRNA 4-thiouridine synthase (EC 2.8.1.4)
284,WP_000872653.1|NWMN_RS09020|NWMN_1609|,NWMN_RS09020,NWMN_1609,thiI,tRNA uracil 4-sulfurtransferase ThiI[Staphylo...,Thiamine; tRNA and rRNA base modification,"""Thiamin biosynthesis""",WP_000872653.1,ACNFN8_03075,thiI,tRNA 4-thiouridine synthase (EC 2.8.1.4),METABOLISM,"Cofactors, Vitamins, Prosthetic Groups",Fe-S clusters,Fe-S_cluster_assembly,Fe-S cluster assembly,tRNA_4-thiouridine_synthase_(EC_2.8.1.4),tRNA 4-thiouridine synthase (EC 2.8.1.4)
285,WP_000872653.1|NWMN_RS09020|NWMN_1609|,NWMN_RS09020,NWMN_1609,thiI,tRNA uracil 4-sulfurtransferase ThiI[Staphylo...,Thiamine; tRNA and rRNA base modification,"""Thiamin biosynthesis""",WP_000872653.1,ACNFN8_03075,thiI,tRNA 4-thiouridine synthase (EC 2.8.1.4),METABOLISM,"Cofactors, Vitamins, Prosthetic Groups",Thiamin,"Thiamin,_thiazole,_hydroxymethylpyrimidine_sal...","Thiamin, thiazole, hydroxymethylpyrimidine sal...",tRNA_4-thiouridine_synthase_(EC_2.8.1.4),tRNA 4-thiouridine synthase (EC 2.8.1.4)
286,WP_000594959.1|NWMN_RS11555|NWMN_1997|,NWMN_RS11555,NWMN_1997,thiD,bifunctional hydroxymethylpyrimidine kinase/p...,Thiamine,"""Thiamin biosynthesis""",WP_000594959.1,ACNFN8_11690,thiD,Hydroxymethylpyrimidine kinase (EC 2.7.1.49) @...,METABOLISM,"Cofactors, Vitamins, Prosthetic Groups",Thiamin,"Thiamin,_thiazole,_hydroxymethylpyrimidine_sal...","Thiamin, thiazole, hydroxymethylpyrimidine sal...",Hydroxymethylpyrimidine_kinase_(EC_2.7.1.49),Hydroxymethylpyrimidine kinase (EC 2.7.1.49)


In [21]:
test_data.to_excel("../data/example_s_aureus_bv_brc_subsystem.xlsx", index=False)

In [25]:
f"{','.join(test_data["wp_id"][:10])}"

'WP_000942300.1,WP_000602071.1,WP_000524836.1,WP_001229077.1,WP_001172340.1,WP_000840574.1,WP_000148605.1,WP_001151902.1,WP_000069282.1,WP_000745871.1'